# `jwst_psfmc` — PSF Photometry on JWST Difference Images

## Complete Demonstration: Non-Detection Upper Limit & Detection Flux Posterior

**Source:** INDEX 43 (NXT44), NEXUS JWST Survey  
**Filter:** F200W (30 mas pixel scale)  
**Epochs:**
- `example1` — the transient is *absent* (non-detection)
- `example2` — the transient is clearly *detected*

---

### Why correlated noise matters

JWST images are produced by the **drizzle algorithm**, which co-adds individual
exposures at sub-pixel accuracy. Each output pixel is a weighted sum of overlapping
detector pixels, so neighbouring output pixels share the same photon counts — they
are **correlated**. If we naïvely treat the pixels as independent in a χ² PSF fit,
the likelihood surface is artificially narrow and the flux uncertainties are
systematically underestimated.  Even in full MCMC fitting with correlated-noise
likelihoods, neglecting the covariance kernel underestimates flux uncertainties by
~30% in F200W (Zhuang et al., *NEXUS: Transient Searches and First Results from
Year One Observations*, in prep.).

`jwst_psfmc` corrects for this by:
1. Measuring the covariance kernel from source-free sky.
2. Evaluating the likelihood in **Fourier space**, where the correlated noise
   becomes a simple per-frequency weighting by the power spectrum of the kernel.
3. Running an **MCMC ensemble sampler** (`emcee`) to obtain the full posterior
   distribution of the four model parameters: `flux, dx, dy, bkg`.

---

### Contents

1. [Setup & data loading](#1.-Setup-&-data-loading)
2. [Covariance kernel inspection](#2.-Covariance-kernel-inspection)
3. [Non-detection: MCMC & upper limit (example1)](#3.-Non-detection:-MCMC-&-3σ-upper-limit-(example1))
4. [Detection: MCMC & flux posterior (example2)](#4.-Detection:-MCMC-&-flux-posterior-(example2))
5. [Saving and reloading results](#5.-Saving-and-reloading-results)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from astropy.io import fits
from astropy.stats import sigma_clipped_stats

import jwst_psfmc as jpm

mpl.rc('xtick', direction='in', top=True)
mpl.rc('ytick', direction='in', right=True)
mpl.rc('font', size=12)

print(f'jwst_psfmc version: {jpm.__version__}')

## 1. Setup & data loading

The example data lives in `data/` and `PSF/` relative to this notebook.
Clone the repository and open the notebook from `examples/` to run it:

```bash
git clone https://github.com/mingyangzhuang/jwst_psfmc.git
cd jwst_psfmc/examples
jupyter notebook demo_psf_photometry.ipynb
```


In [ ]:
# ── difference images and error maps ──────────────────────────────────────
diff_example1  = fits.getdata('data/example1_f200w_diff.fits').astype(np.float64)
err_example1   = fits.getdata('data/example1_f200w_diff_error.fits').astype(np.float64)
diff_example2  = fits.getdata('data/example2_f200w_diff.fits').astype(np.float64)
err_example2   = fits.getdata('data/example2_f200w_diff_error.fits').astype(np.float64)

# ── pre-computed covariance kernels ───────────────────────────────────────
kernel_example1 = fits.getdata('data/example1_f200w_cov_kernel.fits').astype(np.float64)
kernel_example2 = fits.getdata('data/example2_f200w_cov_kernel.fits').astype(np.float64)

# ── 4× oversampled PSF models ─────────────────────────────────────────────
psf_raw_example1 = fits.getdata('PSF/example1_f200w_PSF_4_c.fits').astype(np.float64)
psf_raw_example2 = fits.getdata('PSF/example2_f200w_PSF_4_c.fits').astype(np.float64)

print(f'Difference image shape: {diff_example2.shape}')
print(f'PSF (oversampled) shape: {psf_raw_example2.shape}')
print(f'Covariance kernel shape: {kernel_example2.shape}')

In [ ]:
# ── Extract 9×9 fitting stamps centred on the source ──────────────────────
# The source is at the geometric centre of the cutout.
def extract_stamp(img, half=4):
    cy, cx = img.shape[0] // 2, img.shape[1] // 2
    return img[cy - half : cy + half + 1, cx - half : cx + half + 1]

stamp_example1 = extract_stamp(diff_example1)
err_example1s  = extract_stamp(err_example1)
stamp_example2 = extract_stamp(diff_example2)
err_example2s  = extract_stamp(err_example2)

_, _, rms_example1 = sigma_clipped_stats(diff_example1, sigma=3)
_, _, rms_example2 = sigma_clipped_stats(diff_example2, sigma=3)
max_example1 = np.nanmax(stamp_example1)
max_example2 = np.nanmax(stamp_example2)

print(f'example1 stamp: {stamp_example1.shape},  peak = {max_example1:.4f},  image RMS = {rms_example1:.4f}')
print(f'example2 stamp: {stamp_example2.shape},  peak = {max_example2:.4f},  image RMS = {rms_example2:.4f}')

In [ ]:
# ── Side-by-side overview of the two epochs ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
titles = ['example1 — non-detection', 'example2 — detection']
stamps = [stamp_example1, stamp_example2]

for ax, title, s, rms in zip(axes, titles, stamps, [rms_example1, rms_example2]):
    vmax = 3 * rms
    im = ax.imshow(s, origin='lower', cmap='viridis', vmin=-vmax, vmax=vmax)
    ax.set_title(title, fontsize=13)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label='Flux')

fig.suptitle('Source 43 — F200W difference image stamps (9×9 px)', y=1.02)
fig.tight_layout()
plt.show()

## 2. Covariance kernel inspection

The covariance kernel describes how strongly adjacent pixels are correlated.
For a drizzled F200W JWST mosaic, pixels within ~2–3 px are strongly correlated
due to the drizzle convolution kernel. The `SplitCosineBellWindow` taper ensures
the kernel goes smoothly to zero at the edges, preventing Gibbs ringing in
the Fourier transform.

The **power spectrum** of this kernel is what enters the Fourier-space likelihood:
frequencies where the noise is correlated get downweighted.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Spatial covariance kernels
for ax, k, title in zip(axes[:2],
                        [kernel_example1, kernel_example2],
                        ['example1 covariance kernel', 'example2 covariance kernel']):
    im = ax.imshow(k, origin='lower', cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_title(title, fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# Power spectrum of example2 kernel (what enters the likelihood)
k_padded = np.zeros_like(stamp_example2)
ny, nx = stamp_example2.shape
ky, kx = kernel_example2.shape
y0, x0 = (ny - ky) // 2, (nx - kx) // 2
k_padded[y0:y0+ky, x0:x0+kx] = kernel_example2
ps = np.real(np.fft.fft2(np.fft.ifftshift(k_padded)))
ps = np.clip(ps, 1e-8, None)
im = axes[2].imshow(np.fft.fftshift(np.log10(ps)), origin='lower', cmap='plasma')
axes[2].set_title('log₁₀ power spectrum (example2)', fontsize=12)
axes[2].axis('off')
plt.colorbar(im, ax=axes[2], fraction=0.046, pad=0.04)

fig.suptitle('Pixel-to-pixel covariance structure', y=1.02)
fig.tight_layout()
plt.show()

## 3. Non-detection: Two-run MCMC & 3σ upper limit (example1)

In example1, the transient has not yet brightened. There is no visible source in
the difference image stamp. We perform **two** MCMC runs:

1. **Detection check** — a broad search over all four parameters to confirm the
   source is genuinely absent (flux posterior consistent with zero).
2. **Upper limit** — a second, tighter fit that restricts the centroid search to
   ±0.1 pixels (position known from the detection epoch) and derives the official
   3σ upper limit.

### Why two runs?

The first run uses wide priors on `dx` and `dy` (±1 pixel) so that any
background fluctuation mimicking a source at an unusual position is properly
sampled. This confirms the absence of a detection.

The second run tightens the centroid priors to ±0.1 pixels. Residual small-scale
background fluctuations can mimic low-level source emission at the 1–2σ level,
so the 3σ upper limit is taken as the 99.7th percentile of the posterior flux
distribution — a deliberately conservative bound.


In [ ]:
# ── Run 1: Broad MCMC for detection confirmation ────────────────────────
# Wide priors on dx/dy (±1 pix) ensure any background fluctuation at any
# position in the stamp is properly sampled. This confirms genuine absence.
fit_nd_broad = jpm.prepare_for_fitting(
    data=stamp_example1,
    err=err_example1s,
    psf_os=psf_raw_example1,
    cov_kernel=kernel_example1,
    dx_init=0.0,
    dy_init=0.0,
    oversamp=4,
    native_shape=(11, 11),
    nwalkers=48,
    # default pos_prior_half_width=1.0 → ±1 pix
)

print('Prior bounds (broad, Run 1):')
for k, v in fit_nd_broad['prior_bounds'].items():
    print(f'  {k}: ({v[0]:.3f}, {v[1]:.3f})')
print(f'PSF encircled-energy fraction: {fit_nd_broad["psf_prepare_fraction"]:.4f}')

In [ ]:
# ── Run MCMC: broad (detection check) ───────────────────────────────────
print('Running MCMC (Run 1: broad priors, detection check)...')
sampler_nd_broad = jpm.run_mcmc(
    **fit_nd_broad,
    nsteps=2000,
    ncores=1,        # increase to e.g. 8 for faster runs outside notebooks
    progress=True,
)
print(f'Mean acceptance fraction: {np.mean(sampler_nd_broad.acceptance_fraction):.3f}')

In [ ]:
# ── Convergence diagnostics — Run 1 (broad) ─────────────────────────────
fig_chains_nd_broad, _ = jpm.plot_chains(
    sampler_nd_broad, burnin=500, title='example1 chains — Run 1 (broad)'
)
plt.show()

try:
    tau_nd_broad = sampler_nd_broad.get_autocorr_time(quiet=True)
    print(f'Autocorrelation times: {dict(zip(["flux","dx","dy","bkg"], tau_nd_broad.round(1)))}')
    print(f'Convergence ratio (nsteps/tau, need >50): {(2000 / tau_nd_broad).round(1)}')
except Exception as e:
    print(f'tau not reliable yet: {e}')

In [ ]:
# ── Posterior summary — Run 1 (broad) ───────────────────────────────────
summary_nd_broad = jpm.summarize_emcee(sampler_nd_broad, burnin=500, thin=4)

flat_nd_broad = sampler_nd_broad.get_chain(discard=500, thin=4, flat=True)
flux_samples_nd_broad = flat_nd_broad[:, 0]

print(f"Flux posterior (Run 1 — broad priors):")
print(f"  median = {summary_nd_broad['flux']['median']:.4f}")
print(f"  +1σ    = +{summary_nd_broad['flux']['plus_1sigma']:.4f}")
print(f"  -1σ    = -{summary_nd_broad['flux']['minus_1sigma']:.4f}")

# Confirm non-detection: median flux is consistent with zero within 1σ
is_not_detected = abs(summary_nd_broad['flux']['median']) < summary_nd_broad['flux']['plus_1sigma']
print(f"\nNon-detection confirmed: {is_not_detected}")

In [ ]:
# ── Visualise the flux posterior — Run 1 (broad) ────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(flux_samples_nd_broad, bins=80, density=True, color='steelblue',
        alpha=0.7, label='flux posterior (Run 1, broad)')
ax.axvline(summary_nd_broad['flux']['median'], color='navy', lw=1.8, label='median')
ax.axvline(0, color='k', lw=0.8, linestyle=':')
ax.set_xlabel('Flux')
ax.set_ylabel('Probability density')
ax.set_title('example1 — Run 1 (broad priors) flux posterior')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# ── Corner plot — Run 1 (broad) ─────────────────────────────────────────
fig_corner_nd_broad, _ = jpm.plot_corner(
    sampler_nd_broad, burnin=500, thin=4, title='example1 — Run 1 (broad) posterior'
)
plt.show()

In [ ]:
# ── Triptych at posterior median — Run 1 (broad) ────────────────────────
fig_trip_nd_broad, _ = jpm.plot_psf_fit_triptych(
    stamp_example1, err_example1s,
    fit_nd_broad['psf_os'], oversamp=4,
    native_shape=fit_nd_broad['native_shape'],
    psf_prepare_fraction=fit_nd_broad['psf_prepare_fraction'],
    summary=summary_nd_broad,
    fig_title='Source 43 — example1 F200W  (Run 1: broad)',
)
plt.show()

## 4. Upper limit: Tight MCMC run (example1, Run 2)

With the non-detection confirmed, we now derive the **official 3σ upper limit**.
We restrict the centroid search to ±0.1 pixels (the source position is known from
the detection epoch) and take the 99.7th percentile of the posterior flux
distribution. This guards against false positives from residual background
fluctuations at the 1–2σ level.


In [ ]:
# ── Run 2: Tight MCMC for upper limit ───────────────────────────────────
# Restrict dx/dy to ±0.1 pix since the source position is known from the
# detection epoch. This prevents background fluctuations from biasing the
# upper limit through spurious centroid offsets.
fit_nd_tight = jpm.prepare_for_fitting(
    data=stamp_example1,
    err=err_example1s,
    psf_os=psf_raw_example1,
    cov_kernel=kernel_example1,
    dx_init=0.0,
    dy_init=0.0,
    oversamp=4,
    native_shape=(11, 11),
    nwalkers=48,
    pos_prior_half_width=0.1,  # restrict centroid search to ±0.1 pix
)

print('Prior bounds (tight, Run 2):')
for k, v in fit_nd_tight['prior_bounds'].items():
    print(f'  {k}: ({v[0]:.3f}, {v[1]:.3f})')
print(f'PSF encircled-energy fraction: {fit_nd_tight["psf_prepare_fraction"]:.4f}')

In [ ]:
# ── Run MCMC: tight (upper limit) ───────────────────────────────────────
print('Running MCMC (Run 2: tight priors, upper limit)...')
sampler_nd_tight = jpm.run_mcmc(
    **fit_nd_tight,
    nsteps=2000,
    ncores=1,
    progress=True,
)
print(f'Mean acceptance fraction: {np.mean(sampler_nd_tight.acceptance_fraction):.3f}')

In [ ]:
# ── Convergence diagnostics — Run 2 (tight) ─────────────────────────────
fig_chains_nd_tight, _ = jpm.plot_chains(
    sampler_nd_tight, burnin=500, title='example1 chains — Run 2 (tight)'
)
plt.show()

try:
    tau_nd_tight = sampler_nd_tight.get_autocorr_time(quiet=True)
    print(f'Autocorrelation times: {dict(zip(["flux","dx","dy","bkg"], tau_nd_tight.round(1)))}')
    print(f'Convergence ratio (nsteps/tau): {(2000 / tau_nd_tight).round(1)}')
except Exception as e:
    print(f'tau not reliable yet: {e}')

In [ ]:
# ── Posterior summary and 3σ upper limit — Run 2 (tight) ────────────────
summary_nd_tight = jpm.summarize_emcee(sampler_nd_tight, burnin=500, thin=4)

flat_nd_tight = sampler_nd_tight.get_chain(discard=500, thin=4, flat=True)
flux_samples_nd_tight = flat_nd_tight[:, 0]

# 3σ upper limit at 99.7th percentile. This is deliberately conservative:
# residual small-scale background fluctuations can mimic low-level source
# emission at 1–2σ, so we take the tail of the posterior to guard against
# false positives.
ul_1sigma = np.percentile(flux_samples_nd_tight, 84.1)
ul_2sigma = np.percentile(flux_samples_nd_tight, 97.7)
ul_3sigma = np.percentile(flux_samples_nd_tight, 99.7)

print(f"Flux posterior (Run 2 — tight priors):")
print(f"  median = {summary_nd_tight['flux']['median']:.4f}")
print(f"  +1σ    = +{summary_nd_tight['flux']['plus_1sigma']:.4f}")
print(f"  -1σ    = -{summary_nd_tight['flux']['minus_1sigma']:.4f}")
print()
print(f"  1σ upper limit (84th pct): {ul_1sigma:.4f}")
print(f"  2σ upper limit (97.7th):   {ul_2sigma:.4f}")
print(f"  3σ upper limit (99.7th):   {ul_3sigma:.4f}")

In [ ]:
# ── Visualise posteriors: Run 1 (broad) vs Run 2 (tight) ────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(flux_samples_nd_broad, bins=80, density=True, color='steelblue',
        alpha=0.6, label='Run 1 (broad priors)')
ax.hist(flux_samples_nd_tight, bins=80, density=True, color='darkorange',
        alpha=0.6, label='Run 2 (tight priors, ±0.1 pix)')
ax.axvline(0, color='k', lw=0.8, linestyle=':')
ax.axvline(ul_3sigma, color='darkorange', lw=1.5, linestyle='--',
           label=f'Run 2 3σ UL = {ul_3sigma:.4f}')
ax.set_xlabel('Flux')
ax.set_ylabel('Probability density')
ax.set_title('Source 43 F200W — flux posteriors: broad vs tight')
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
# ── Corner plot — Run 2 (tight) ─────────────────────────────────────────
fig_corner_nd_tight, _ = jpm.plot_corner(
    sampler_nd_tight, burnin=500, thin=4, title='example1 — Run 2 (tight) posterior'
)
plt.show()

In [ ]:
# ── Triptych: Data / Model / Residual — Run 2 (tight) ───────────────────
fig_trip_nd_tight, _ = jpm.plot_psf_fit_triptych(
    stamp_example1, err_example1s,
    fit_nd_tight['psf_os'], oversamp=4,
    native_shape=fit_nd_tight['native_shape'],
    psf_prepare_fraction=fit_nd_tight['psf_prepare_fraction'],
    summary=summary_nd_tight,
    fig_title='Source 43 — example1 F200W  (Run 2: tight, 3σ UL)',
)
plt.show()

### Interpreting the two runs

- **Run 1 (broad)** — wide dx/dy priors (±1 pix) confirm the source is genuinely
  absent. The flux posterior is consistent with zero, and the centroid posteriors
  show no coherent signal.
- **Run 2 (tight)** — restricted dx/dy priors (±0.1 pix) yield the official 3σ
  upper limit. The tighter centroid constraint narrows the flux posterior tail,
  producing a more meaningful upper bound.


## 6. Saving and reloading results

Results are stored as compressed `.npz` archives. The `load_emcee_results`
function returns a clean dictionary so you don't need to remember
the internal key names.


In [ ]:
import tempfile, os

with tempfile.TemporaryDirectory() as tmpdir:
    path_nd_broad  = os.path.join(tmpdir, 'example1_f200w_run1_broad')
    path_nd_tight  = os.path.join(tmpdir, 'example1_f200w_run2_tight')
    path_det       = os.path.join(tmpdir, 'example2_f200w')

    payload_nd_broad  = jpm.save_emcee_results(path_nd_broad,  sampler_nd_broad, burnin=500, thin=4)
    payload_nd_tight  = jpm.save_emcee_results(path_nd_tight,  sampler_nd_tight, burnin=500, thin=4)
    payload_det       = jpm.save_emcee_results(path_det,       sampler_det,      burnin=500, thin=4)

    res_nd_broad  = jpm.load_emcee_results(path_nd_broad  + '.npz')
    res_nd_tight  = jpm.load_emcee_results(path_nd_tight  + '.npz')
    res_det       = jpm.load_emcee_results(path_det       + '.npz')

print('Loaded keys:', list(res_det.keys()))
print('\nRun 2 (tight) summary (reloaded):')
for k, v in res_nd_tight['summary'].items():
    print(f"  {k:6s} = {v['median']:+.4f}  +{v['plus_1sigma']:.4f} / -{v['minus_1sigma']:.4f}")
print(f'\n3σ upper limit: {ul_3sigma:.4f}')
print(f'Acceptance fraction: {res_det["acceptance_fraction"].mean():.3f}')
print(f'Chain τ: {res_det["tau"].round(1)}')
print(f'τ converged: {res_det["tau_ok"]}')

---

## Summary

| Quantity | example1 Run 1 (broad) | example1 Run 2 (tight) | example2 (detection) |
|----------|---------------------|----------------------|------------------|
| dx/dy prior | ±1.0 pix | ±0.1 pix | ±1.0 pix |
| Flux posterior median | ≈ 0 | ≈ 0 | > 0 (significant) |
| Detection significance | < 1σ | — | > 5σ |
| 3σ upper limit | — | `ul_3sigma` | — |
| Residual structure | noise-level | noise-level | noise-level |

The two-run workflow ensures the non-detection is confirmed with broad priors
before deriving a tight upper limit. The correlated-noise likelihood correctly
accounts for pixel correlations throughout.

---

*Notebook generated with `jwst_psfmc` — see [README](../README.md) for installation.*